# 第55课 · 给计算图装上关节——`__pow__` / `relu` / `tanh`，前向传播（forward pass）算子集齐

**目标**：为 `Value` 补全 `__pow__` / `relu` / `tanh` 等，让前向算子集齐。

> 每个 op：局部导数一行 + 小图；为 L56 `backward` 准备关节。

🔗 **Aurora 连接**：这些算子对应神经网络里的每一层——线性变换（`+`、`*`、`**`）加激活函数（activation function，`tanh`/`relu`），自动微分就建立在这套算子之上。这些算子未来将落到 Aurora ML 引擎（`src/aurora/ml/`，计划中模块）里。

← **上一课**　[L54 · Value 计算图](L54_value_autograd.ipynb)

> 上节课学习了 **Value 计算图**：标量自动微分：前向值 + 反向梯度，手写 add / mul 节点。  
> 本课将探讨 **Value 算子补全**。

## 开篇回调：L54 的 `Value` 还缺什么？

L54 已经把 `__add__` / `__mul__` 的梯度规则说清了，但还缺三类“节点动作”：`__pow__`、`relu`、`tanh` / `exp`。
本课只做一件事：把这些算子补齐，让同一套计算图能表达真正的神经网络前向。

## 本课剧情：为什么神经网络能用同一套代码同时学"加法"和"非线性"？

想象你在搭积木：`z = w1*x1 + w2*x2 + b`，然后 `output = tanh(z)`。这看起来只是两行数学，但训练时需要知道"改变 w1 多少，loss 会变多少"——对每个积木都要求偏导数。

关键洞察：**每个算子就是一个积木，知道自己怎么把梯度传回去**。

```
前向：out = a * b           → out.data = a.data * b.data
反向：∂L/∂a = ∂L/∂out · b  → a.grad += out.grad * b.data
     ∂L/∂b = ∂L/∂out · a  → b.grad += out.grad * a.data
```

每个算子在完成前向计算的同时，把"怎么反向传梯度"封装进一个 **_backward 闭包**（closure）——等到 `L.backward()` 被调用时，从输出节点逆向遍历 DAG，逐节点调用 `_backward()`。

**三个新算子**：

| 算子 | 前向 | 导数（解析式） |
|---|---|---|
| `__pow__(n)` | `x^n` | `n·x^{n-1}` |
| `relu(x)` | `max(0,x)` | `1 if x>0 else 0`（次梯度）|
| `tanh(x)` | `(e²ˣ-1)/(e²ˣ+1)` | `1 - tanh²(x)`（可用前向输出直接算）|

本节任务：补全这三个算子，让感知机（perceptron） `z=w1x1+w2x2+b`，`output=tanh(z)` 端到端反向传播正确。

In [ ]:
import math

# 完整的 Value 类（含加法、乘法基础，本节要补全其余算子）
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    # --- 已有算子 ---
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad  += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __radd__(self, other): return self + other

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad  += other.data * out.grad
            other.grad += self.data  * out.grad
        out._backward = _backward
        return out

    def __rmul__(self, other): return self * other

    def __neg__(self):   return self * -1
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return Value(other) + (-self)
    def __truediv__(self, other):
        # 注意：依赖 __pow__。请先在下方练习格实现 __pow__，否则此处会抛出 NotImplementedError
        return self * other**-1
    def __rtruediv__(self, other):
        # 同上，依赖 __pow__
        return Value(other) * self**-1

    # --- 待实现算子（占位，下面编码任务格会替换）---
    def __pow__(self, n): raise NotImplementedError
    def relu(self):       raise NotImplementedError
    def tanh(self):       raise NotImplementedError
    def exp(self):        raise NotImplementedError

    # --- 反向传播（拓扑序） ---
    def backward(self):
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for c in v._prev: build(c)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

print("Value 类已定义")

## 深挖：.backward() 里的"拓扑排序"是什么？

你可能看到了 Value 类里这段代码：

```python
def backward(self):
    topo, visited = [], set()
    def build(v):
        if v not in visited:
            visited.add(v)
            for c in v._prev: build(c)  # 递归遍历子节点
            topo.append(v)
    build(self)
    self.grad = 1.0
    for v in reversed(topo):
        v._backward()
```

这里的"拓扑排序"听起来很复杂，但核心思想其实很直观。

**第一步：理解计算图的"依赖关系"**

想象你要建一所房子。先打地基，再建墙，再装门窗，最后装修。这些步骤有个顺序——你不能在没打地基时就装门窗。

计算图也是这样。例如：
```
x = Value(2.0)
y = Value(3.0)
z = x * y        # z 依赖 x 和 y
w = z + x        # w 依赖 z 和 x
```

这里 `z` 依赖 `x` 和 `y`，`w` 依赖 `z`。我们说 `z` 是 `w` 的"父节点"。

**第二步：为什么要反向遍历？**

反向传播就是从"输出"（`w`）往"输入"（`x`, `y`）推梯度。但有个规则：**只有当一个节点的所有上游节点的梯度都算好了，这个节点才能计算自己的梯度**。

错误的方式（直接从输出开始遍历）：
```
w.backward()  # 计算 w.grad 相对于 z 和 x 的梯度
  → 需要 z.grad，但还没算！
```

正确的方式（逆拓扑排序）：
```
1. 先计算 x.grad（叶节点）
2. 再计算 z.grad（z 依赖 x）
3. 最后计算 w.grad（w 依赖 z 和 x）
```

**第三步：代码的机制**

```python
def build(v):
    if v not in visited:          # 避免重复访问
        visited.add(v)
        for c in v._prev:         # 遍历 v 的所有输入（子节点）
            build(c)
        topo.append(v)            # 只有在所有子节点都处理过，才把 v 加入
```

这个递归确保：**越往下游的节点，越先被加入 `topo`**。再用 `reversed()` 反过来，就是"从输出逆向到输入"的顺序。

举例：
```
计算图：x → z → w
build(w):
  build(z):  （z 还没访问过）
    build(x):（x 还没访问过）
      （x 没有子节点，立即 topo.append(x)）
    topo.append(z)
  topo.append(w)

结果：topo = [x, z, w]
reversed(topo) = [w, z, x]  ← 正好是反向传播的顺序！
```

**实践意义**

这个"拓扑排序"确保梯度正确累积。如果顺序错了（比如先算 `x.grad`，再算 `z.grad`），梯度公式就会出错。

## 概念 1：每个算子 = 新节点 + _backward 闭包

调用 `a * b` 时发生两件事：

1. **前向**：`out = Value(a.data * b.data)`，挂载 `_prev = {a, b}`
2. **后向闭包**：在 `_backward` 里写入链式法则——`a.grad += b.data * out.grad`

闭包捕获了 `a`、`b`、`out` 三个活对象，调用 `.backward()` 时按拓扑逆序执行，梯度自动累积到叶节点的 `.grad`。

### 🔍 深挖：闭包（closure）是如何"记住"变量的？

如果你是第一次看到这样的代码，可能会疑惑：为什么 `_backward` 函数定义在 `__mul__` 里，后来却还能访问那时的 `self`、`other`、`out`？

**直观例子**：比较两种方式

```python
# ❌ 错误方式：直接存数值
def __mul__(self, other):
    out = Value(self.data * other.data)
    b_val = other.data  # 存"当时"other 的数值
    def _backward():
        # ⚠️ 问题：如果后来 other.grad 被修改了，这里看不到！
        self.grad += b_val * out.grad  # 只知道数值，丢失了对象
    return out

# ✅ 正确方式：闭包捕获对象引用
def __mul__(self, other):
    out = Value(self.data * other.data)
    def _backward():
        # ✅ 闭包"记住"了 self、other、out 三个对象本身
        # 即使后来调用 .backward()，看到的是这些对象当时的状态
        self.grad += other.data * out.grad
    out._backward = _backward
    return out
```

**为什么需要闭包（而不是直接存数值）？**

反向传播时，`_backward()` 被调用时，梯度已经通过 `out.grad` 流入了下游节点。此时读取 `other.data`，得到的仍是"前向时"的值（因为前向后参数不变，只有梯度在反向时累积）。闭包让我们延迟执行"读取 `other.data`"这个动作，从"定义时"推迟到"调用时"——这正是我们需要的。

换句话说：
- 定义闭包 → 记住"这些变量存在"
- 调用闭包 → 读取"现在这些变量的值"

这也是 Python 自动微分库（如 PyTorch）的核心机制。

In [ ]:
# 演示：加法节点的 _backward
x = Value(3.0)
y = Value(4.0)
z = x + y       # z.data = 7.0
z.grad = 1.0
z._backward()   # 手动触发
print(f"x.grad = {x.grad}, y.grad = {y.grad}")  # 应该都是 1.0

## 补充：幂函数的导数公式是怎么来的？

在实现 `__pow__` 前，让我们推导一遍 `d(x^n)/dx = n·x^(n-1)` 这个公式，这样你才能真正理解和记住它（而不是死背）。

**情景 1：整数幂（最简单）**

对于 `f(x) = x³`，用导数定义：
```
f'(x) = lim[h→0] (f(x+h) - f(x)) / h
      = lim[h→0] ((x+h)³ - x³) / h
```

展开 `(x+h)³` 用二项式：
```
(x+h)³ = x³ + 3x²h + 3xh² + h³
```

代入：
```
f'(x) = lim[h→0] (x³ + 3x²h + 3xh² + h³ - x³) / h
      = lim[h→0] (3x²h + 3xh² + h³) / h
      = lim[h→0] (3x² + 3xh + h²)    ← 约去 h
      = 3x²   ← 当 h→0 时，3xh和h²都消失
```

看出规律：`x^3` 的导数是 `3x²`（系数是原指数 3，新指数是 3-1=2）。

**推广到任意整数幂 n**

用类似的方法（或用二项式定理），可以证明：
```
d(x^n)/dx = n·x^(n-1)
```

验证几个例子：
- `x¹` → 导数 `1·x^0 = 1`  ✓（常函数导数为 0，线性导数为 1）
- `x²` → 导数 `2·x^1 = 2x`  ✓
- `x^(-1) = 1/x` → 导数 `-1·x^(-2) = -1/x²`  ✓（符合商法则结果）

**为什么对负幂也成立？**

虽然推导里用了二项式展开，但这套公式对负整数也适用。例如 `x^(-2)`：
```
d(x^(-2))/dx = -2·x^(-3) = -2/x³
```

这和用商法则对 `1/x²` 求导的结果一致。

**本课应用**

在 `__pow__` 的 `_backward` 中，我们就用这个公式：
```python
self.grad += n * (self.data ** (n-1)) * out.grad
```

其中 `n·self.data^(n-1)` 是"局部导数"，乘以 `out.grad`（上游梯度）就得到这一步对输入的贡献。

## 补充：tanh 导数公式 `1 - tanh²(x)` 是如何推导的？

在理解"为什么 tanh 梯度可以用前向输出直接计算"之前，我们先推导一遍 `d(tanh(x))/dx = 1 - tanh²(x)` 这个公式。

**第一步：回顾 tanh 的定义**

```
tanh(x) = (e^x - e^(-x)) / (e^x + e^(-x))
        = (e^(2x) - 1) / (e^(2x) + 1)    ← 两种等价写法，我们用第二种
```

**第二步：用商法则求导**

回忆商法则：`d/dx [f/g] = (f'g - fg') / g²`

这里：
- `f = e^(2x) - 1` → `f' = 2e^(2x)`
- `g = e^(2x) + 1` → `g' = 2e^(2x)`

代入：
```
d(tanh)/dx = (2e^(2x) · (e^(2x) + 1) - (e^(2x) - 1) · 2e^(2x)) / (e^(2x) + 1)²
```

分子展开：
```
= (2e^(2x) · e^(2x) + 2e^(2x) - 2e^(2x) · e^(2x) + 2e^(2x)) / (e^(2x) + 1)²
= (2e^(2x) + 2e^(2x)) / (e^(2x) + 1)²
= 4e^(2x) / (e^(2x) + 1)²
```

**第三步：用 tanh 的值来重新表达**

现在，令 `t = tanh(x) = (e^(2x) - 1) / (e^(2x) + 1)`。

从定义 `t = (e^(2x) - 1) / (e^(2x) + 1)` 可以推导出：
```
e^(2x) - 1 = t(e^(2x) + 1)
e^(2x) - 1 = t·e^(2x) + t
e^(2x) - t·e^(2x) = 1 + t
e^(2x)(1 - t) = 1 + t
e^(2x) = (1 + t) / (1 - t)
```

同时，从 `e^(2x) + 1 = (e^(2x) - 1 + 2) = t(e^(2x) + 1) + 2`... 其实有更简洁的方法：

我们知道 `e^(2x) + 1 = e^(2x) + 1`，从上面 `e^(2x) = (1+t)/(1-t)` 代入：
```
e^(2x) + 1 = (1+t)/(1-t) + 1 = (1+t + 1-t)/(1-t) = 2/(1-t)
```

所以：
```
d(tanh)/dx = 4e^(2x) / (e^(2x) + 1)²
           = 4 · (1+t)/(1-t) / (2/(1-t))²
           = 4 · (1+t)/(1-t) · (1-t)²/4
           = (1+t)(1-t)
           = 1 - t²
```

**结论**

```
d(tanh(x))/dx = 1 - tanh²(x)
```

这说明什么？导数只依赖于 `tanh(x)` 本身的值，而不需要知道原始输入 `x` 是多少！这就是为什么在代码里可以写成 `(1 - out.data**2) * out.grad`。

## 概念 2：tanh 梯度用前向输出值直接表达

`tanh(x) = (e^{2x} - 1) / (e^{2x} + 1)`，其导数：

```
d(tanh(x))/dx = 1 - tanh(x)^2
```

实现技巧：`out.data` 就是已经算好的 `tanh(x)` 值，反向直接写：

```python
self.grad += (1 - out.data**2) * out.grad
```

**为什么能只用前向输出值，不用原始输入 x？**

这是 tanh 的一个数学特性：导数恰好能用**函数值本身**表达。

具体来说：
- 前向时，我们计算了 `out.data = tanh(x)`，存在节点 `out` 里
- 反向时，我们需要计算梯度 `d(tanh(x))/dx = 1 - tanh(x)²`
- 关键：我们不需要知道 `x` 的具体数值！只需要知道 `tanh(x)` 的值
- 用 `out.data` 替代 `tanh(x)`：`1 - out.data²` 就是导数值

这样的好处：
1. 省计算量：不用再调一次 `math.tanh(x)`
2. 节省存储：反向闭包不用特别保存 `x`，只靠 `out.data` 够用
3. 通用性：对于任何激活函数 `f`，只要导数 `f'` 能用 `f(x)` 表达，就能用这个技巧

相比之下，其他激活函数可能没这么幸运。例如 `sigmoid(x) = 1/(1+e^(-x))` 的导数也是 `sigmoid(x)·(1-sigmoid(x))`，也能用前向值直接算。但有些函数就不行——例如 `elu(x) = x if x>0 else alpha·(e^x-1)` 的导数需要知道 `x` 是正是负，光看输出值看不出来。

In [ ]:
# 验证公式：数值差分 vs 解析梯度
import math

x_val = 0.8
h = 1e-5
analytic = 1 - math.tanh(x_val)**2
numeric  = (math.tanh(x_val + h) - math.tanh(x_val - h)) / (2*h)
print(f"解析梯度: {analytic:.6f}")
print(f"数值梯度: {numeric:.6f}")
print(f"误差: {abs(analytic - numeric):.2e}")

## 概念 3：ReLU 梯度与"次梯度"（subgradient）

`relu(x) = max(0, x)`，在 `x > 0` 时梯度为 1，`x < 0` 时梯度为 0，`x = 0` 处次梯度取 0：

```
d(relu(x))/dx = 1  if x > 0 else 0
```

反向实现：

```python
self.grad += (out.data > 0) * out.grad
```

`out.data > 0` 是布尔值，Python 里等于 `1` 或 `0`，直接相乘即可。

**为什么 ReLU 在 x=0 处"不可导"，却非要选一个梯度值？**

ReLU 在 `x=0` 这个点有个"尖角"——函数图像从下面的直线（`y=0`）突然变成上面的直线（`y=x`）。这意味着：
- 从左边接近 0：梯度是 0（因为 `x<0` 区间输出恒为 0）
- 从右边接近 0：梯度是 1（因为 `x>0` 区间输出是 `x` 本身）

在高等数学中，这种情况叫"非光滑"（not smooth），传统的"导数"在这一点没有定义。但我们引入"**次梯度**"这个概念：对于这样的非光滑函数，在尖点处，梯度可以取左右梯度之间的任意值。

**为什么实践中选择 0（而不是 1 或其他值）？**

简单的原因：
1. **概率论角度**：在真实数据中，某个神经元的输入恰好等于 0 的概率接近于 0（几乎不会发生）
2. **稳定性**：选 0 很保守——不传播梯度，这个输入就"断了"；选 1 的话，在正数区行为一致；两者效果差异微乎其微
3. **简化实现**：代码里就是 `(out.data > 0) * out.grad`，当 `out.data == 0` 时自动得 0，非常简洁

所以在实现里，我们就选 0，这既不影响训练效果（因为 0 出现的概率极小），又让代码简洁明了。

**补充：Dead ReLU 问题**

ReLU 的缺点是 `x < 0` 区间梯度始终为 0，这会导致某些神经元永远不被激活（gradient 永远不流向它）。这被称为"Dead ReLU"问题。Leaky ReLU 和 ELU 是改进方案：
- Leaky ReLU：`x<0` 时梯度取小正数（如 0.01），而不是 0
- ELU：负数区用平滑的指数函数，避免硬截断

In [ ]:
# 演示 ReLU 次梯度
vals = [-2.0, 0.0, 3.0]
for v in vals:
    grad = float(v > 0)
    print(f"relu'({v:+.1f}) = {grad}")

## 1. ✏️ 实现 `Value.__pow__`, `Value.relu`, `Value.tanh`, `Value.exp`

**三步实现模板**（每个算子相同结构）：

```python
def __pow__(self, n):
    out = Value(self.data ** n, _children=(self,), _op=f'**{n}')
    def _backward():
        self.grad += n * (self.data ** (n-1)) * out.grad
    out._backward = _backward
    return out
```

**四个算子的 _backward 公式**：

| 算子 | 前向 | 反向梯度公式 | 说明 |
|---|---|---|---|
| `__pow__(n)` | `x^n` | `n · x^(n-1) · out.grad` | 幂法则 |
| `relu` | `max(0, x)` | `(x > 0) · out.grad` | 次梯度（在0处取0） |
| `tanh` | `tanh(x)` | `(1 - tanh²(x)) · out.grad` | 用前向输出直接算 |
| `exp` | `e^x` | `e^x · out.grad` | 导数就是自己 |

**exp() 的用途**

虽然在这节课里 `exp` 不直接用到（我们用 `math.tanh` 的库函数），但为了完整性和后续扩展，我们也实现它。在某些场景下，你可能想自己从指数函数开始构建 tanh：
```
tanh(x) = (e^(2x) - 1) / (e^(2x) + 1)
        = 2*e^(2x) / (e^(2x) + 1) - 1
```
这时就需要 `exp` 算子。另外，softmax、cross-entropy 等损失函数的反向也会用到 exp。

**验收标准**：
- `Value(-2).__pow__(2).data == 4.0`
- `Value(-1.0).relu().data == 0.0`，`Value(2.0).relu().data == 2.0`
- 感知机 `backward()` 后，数值差分误差 < 1e-5
- `Value(1.0).exp().data` 约等于 2.718（自然常数 e）

In [ ]:
# ✏️ TODO：在下面完整实现四个算子，替换 Value 类里的占位 raise NotImplementedError

def __pow__(self, n):
    # ✏️ TODO
    assert isinstance(n, (int, float))
    t   = ...                        # ✏️ TODO: self.data**n
    out = Value(t, (self,), f'**{n}')  # _prev 和 _op 已预填，只需填 t
    def _backward():
        self.grad += ...    # ✏️ TODO: n * self.data**(n-1) * out.grad
    out._backward = _backward
    return out

def relu(self):
    t = ...                 # ✏️ TODO: max(0, self.data)
    out = Value(t, (self,), 'relu')
    def _backward():
        self.grad += ...    # ✏️ TODO: (t > 0) * out.grad
    out._backward = _backward
    return out

def tanh(self):
    t = ...                 # ✏️ TODO: math.tanh(self.data)
    out = Value(t, (self,), 'tanh')
    def _backward():
        self.grad += ...    # ✏️ TODO: (1 - t**2) * out.grad
    out._backward = _backward
    return out

def exp(self):
    t = ...                 # ✏️ TODO: math.exp(self.data)
    out = Value(t, (self,), 'exp')
    def _backward():
        self.grad += ...    # ✏️ TODO: t * out.grad
    out._backward = _backward
    return out

# 绑定到 Value 类
Value.__pow__ = __pow__
Value.relu    = relu
Value.tanh    = tanh
Value.exp     = exp

In [ ]:
# 检查：基本前向值
assert Value(-2.0).__pow__(2).data == 4.0,   "__pow__ 前向错误"
assert Value(-2.0).relu().data == 0.0,        "relu 前向：负数应输出 0"
assert Value( 0.5).relu().data == 0.5,        "relu 前向：正数应透传"
assert abs(Value(1.0).tanh().data - 0.7616) < 1e-4, "tanh(1.0) 应 ≈ 0.7616"
assert abs(Value(1.0).exp().data  - math.e)  < 1e-5, "exp(1.0) 应 ≈ e"

# 检查：__pow__ 反向
a = Value(3.0)
b = a**2          # b = 9, db/da = 2*3 = 6
b.backward()
assert abs(a.grad - 6.0) < 1e-9, f"__pow__ 反向梯度应为 6.0，得到 {a.grad}"

# 检查：relu 反向（正数区）
a = Value(2.0)
r = a.relu()
r.backward()
assert a.grad == 1.0, "relu 正数区反向梯度应为 1.0"

# 检查：relu 反向（负数区）
a = Value(-1.0)
r = a.relu()
r.backward()
assert a.grad == 0.0, "relu 负数区反向梯度应为 0.0"

# 检查：tanh 反向
a = Value(0.0)
t = a.tanh()       # tanh(0)=0, 梯度=1-0^2=1
t.backward()
assert abs(a.grad - 1.0) < 1e-9, "tanh(0) 反向梯度应为 1.0"

print("✅ 所有算子前向 & 反向检查通过")

## 参数实验：二维感知机前向传播与链式法则验证

用 `Value` 算子搭建一个最小感知机：

```
z = w1*x1 + w2*x2 + b
output = tanh(z)         # 激活函数
```

参数：`w1=0.5, w2=-0.3, b=0.1, x1=1.0, x2=2.0`

**预期现象**：
- `z.data` 应为 `0.5*1 + (-0.3)*2 + 0.1 = 0.0`
- `output.data` 应为 `tanh(0.0) = 0.0`
- 调用 `output.backward()` 后，梯度会按"链式法则"逆向流回参数

**链式法则的具体推导**

在这个例子中，让我们手推 `w1` 的梯度。设 loss = `output`（为简单起见，假设我们关心的就是 output 这个值）。

**插播：为什么下面的推导只有两项，标准链式法则不是三项吗？**

如果你在别的地方见过链式法则，可能见过"完整版"写法：

```
∂L/∂w = (∂L/∂output) · (∂output/∂z) · (∂z/∂w)
```

这里 `L` 是"损失函数"（loss）——训练时我们真正想要最小化的那个数字。这三项对应计算图上连续三步跳跃：`w 变化 → z 跟着变 → output 跟着变 → L 跟着变`，然后把每一步的"敏感度"连乘起来，就是 `w` 对 `L` 的总敏感度。

打个比方：你想知道"你的加薪对你今晚心情的影响"，链条是"加薪 → 这个月存款 → 心情"。但如果你今晚的心情就是直接盯着存款数字看，没有任何额外的心理加工（比如没有"跟别人比较"这种中间环节），那"存款对心情的影响"这一项就是 1——存款多 1 块，心情（用存款衡量）就正好多算 1 块，不多不少，因为心情本身**就是**存款数字。

回到公式：本课的感知机例子里，我们还没有引入损失函数——直接让 `loss = output`，也就是"不对 output 做任何额外加工，直接把它当成我们关心的量"。数学上，这叫**恒等函数**（identity function）：`L(output) = output`。恒等函数的导数是多少？回忆 `y = x` 这条直线，斜率恒为 1，所以：

```
∂L/∂output = ∂(output)/∂output = 1
```

把这个 1 代回三项链式法则：

```
∂L/∂w = 1 · (∂output/∂z) · (∂z/∂w) = (∂output/∂z) · (∂z/∂w)
```

——这就是为什么下面的推导直接写成两项：不是漏掉了第一项，而是第一项算出来恰好是 1，乘上 1 不改变结果，所以"隐身"了。

> **提醒**：这只是本课示例的简化！一旦后面的课程接上真正的损失函数（比如均方误差 MSE 或交叉熵 cross-entropy），`∂L/∂output` 就不再等于 1 了，三项链式法则会重新"缺一不可"——到时候你需要先把这一项算出来，再乘回去。L56 会讲到这一步。

通过"计算图"的路径：
```
w1 --→ (w1*x1) --→ (w1*x1 + w2*x2 + b) --→ tanh(...) --→ output
                   (即 z)                    (即 output)
```

链式法则说：`∂output/∂w1 = (∂output/∂z) · (∂z/∂w1)`（这里 `output` 就是 `L`，所以省去了恒为 1 的 `∂L/∂output` 这一项）

分解：
- `∂output/∂z`：output = tanh(z)，所以 `∂output/∂z = tanh'(z) = 1 - tanh²(z) = 1 - 0² = 1`
- `∂z/∂w1`：z = w1*x1 + ..., 所以 `∂z/∂w1 = x1 = 1`

因此：`∂output/∂w1 = 1 · 1 = 1`

同理，`w2` 的梯度：
- `∂output/∂w2 = (∂output/∂z) · (∂z/∂w2) = tanh'(z) · x2 = 1 · 2 = 2`
- `b` 的梯度：`∂output/∂b = tanh'(z) · 1 = 1`

**关键观察**：
1. 梯度从"输出"反向流回"输入"
2. 每一步乘以"局部梯度"（这一步的导数）
3. 代码里的 `.backward()` 就是自动按这个规则逆向遍历计算图，累加梯度

**对比 relu 激活**

如果把 `tanh` 换成 `relu`，在 `z=0` 处，`relu'(0) = 0`（我们的次梯度选择），那么所有梯度都会变 0——路径被"断开"了。这就是 dead ReLU 现象。

In [ ]:
# 二维感知机前向 + 反向
w1 = Value(0.5)
w2 = Value(-0.3)
b  = Value(0.1)
x1 = Value(1.0)
x2 = Value(2.0)

z      = w1*x1 + w2*x2 + b   # 线性组合
output = z.tanh()              # tanh 激活

print(f"z.data     = {z.data:.4f}      (期望 0.0)")
print(f"output.data= {output.data:.4f}  (期望 tanh(0.0) = 0.0)")

output.backward()
print(f"\n反向传播后：")
print(f"w1.grad = {w1.grad:.4f}   (= x1 * tanh'(z) = 1.0 * 1.0)")
print(f"w2.grad = {w2.grad:.4f}   (= x2 * tanh'(z) = 2.0 * 1.0)")
print(f"b.grad  = {b.grad:.4f}   (= tanh'(z) = 1.0)")

# 对比 relu 激活（新建独立输入，避免与上方 tanh 演示的梯度累积）
w1r, w2r, br = Value(0.5), Value(-0.3), Value(0.1)
x1r, x2r = Value(1.0), Value(2.0)   # ← 新建，不复用上方 x1/x2
zr = w1r*x1r + w2r*x2r + br
out_r = zr.relu()
out_r.backward()
print(f"\nrelu 激活：output={out_r.data:.4f}, w1.grad={w1r.grad:.4f}, w2.grad={w2r.grad:.4f}")
print("（relu 在 z=0 处次梯度取 0，梯度传播截断）")

## 本课收束

本节实现了 `__pow__`、`relu`、`tanh`、`exp` 四个算子，每个算子在返回新 `Value` 节点的同时封装了反向梯度闭包。这些算子的输出 `out.data` 就是下一节将要累积梯度的"局部变量"，梯度通过 `out.grad` 向上游流动。完整的算子集让 `Value` 能描述任意深度的计算图——对应 Aurora ML 引擎（`src/aurora/ml/`，计划中模块）里每一层的前向计算。下一节（**L56**）将把 `.backward()` 拆开手推：对整张多层计算图做拓扑排序，逐节点验证「梯度 = 局部梯度 × 上游梯度」的链式传播。

## ✏️ 白板挑战：算子梯度手推（目标 10 分钟）

盖上屏幕，纸上推导：

**问 1**：`z = x**3` 在 `x=2` 处的梯度是多少？  
（导数：3x² → 3×4 = 12）

**问 2**：`tanh` 的导数为什么可以用前向输出值表达？  
（`d/dx tanh(x) = 1 - tanh²(x)`，不需要原始 x，只需 `out.data`）

**问 3**：对计算图 `L = relu(x*w + b)`，`x=1, w=2, b=-1`：
- 前向值：`z = x*w+b = ?`，`L = relu(z) = ?`
- 反向：`∂L/∂w = ?`（注意 relu 在 z>0 时梯度=1）

**问 4**：为什么 ReLU 在 `x=0` 处用次梯度 0（而不是 undefined）？  
（实践中 0 处概率为零，用 0 不影响训练稳定性）

**问 5**：`c = a**2 + b**2`，设 `a=3, b=4`，`c.backward()` 后 `a.grad` 和 `b.grad` 各是多少？  
（`∂c/∂a = 2a = 6`，`∂c/∂b = 2b = 8`）

推导完成后运行下面格对答案。

In [ ]:
# ✏️ 对答案格
import math

# 说明：算子还没实现时，TODO 里的 `...` 会让 Value(...) 抛 TypeError，
# 占位方法则抛 NotImplementedError——两种都按「待实现」处理，先给出文字答案。

# 问1：z=x**3, x=2 → grad=12
try:
    x = Value(2.0)
    z = x.__pow__(3)
    z.grad = 1.0
    z._backward()
    assert abs(x.grad - 12.0) < 1e-12, f"x.grad={x.grad}"
    print(f"Q1 ✅  d/dx(x³)|x=2 = {x.grad:.1f} = 3×2² = 12")
except (NotImplementedError, TypeError):
    print(f"Q1 ✅  d/dx(x³)|x=2 = 3×2²=12（__pow__ 待实现）")

# 问2：tanh 导数用前向输出
x_val = 0.8
tanh_val = math.tanh(x_val)
grad_analytic = 1 - tanh_val**2
grad_numeric  = (math.tanh(x_val+1e-5) - math.tanh(x_val-1e-5)) / (2e-5)
assert abs(grad_analytic - grad_numeric) < 1e-9
print(f"Q2 ✅  tanh'({x_val})=1-tanh²={grad_analytic:.6f}，数值差分={grad_numeric:.6f}")

# 问3：relu(x*w+b), x=1, w=2, b=-1
try:
    x, w, b = Value(1.0), Value(2.0), Value(-1.0)
    z = x*w + b       # z.data = 1.0
    L = z.relu()      # L.data = 1.0 (z>0)
    L.backward()
    assert abs(L.data - 1.0) < 1e-12
    assert abs(w.grad - 1.0) < 1e-12, f"w.grad={w.grad}"
    print(f"Q3 ✅  z=x*w+b={z.data:.1f}>0，L=relu(z)={L.data:.1f}，∂L/∂w={w.grad:.1f}=x=1")
except (NotImplementedError, TypeError):
    print(f"Q3 ✅  x=1,w=2,b=-1: z=1>0，L=relu(z)=1，∂L/∂w=x=1（relu 待实现）")

# 问4：relu 次梯度（概念验证）
relu_grads = {-1.0: 0, 0.0: 0, 1.0: 1}
for v, expected in relu_grads.items():
    got = float(v > 0)
    assert got == expected
print(f"Q4 ✅  relu 次梯度: x<0→0, x=0→0, x>0→1（次梯度，实践中安全）")

# 问5：c=a²+b², a=3, b=4
try:
    a, b = Value(3.0), Value(4.0)
    c = a**2 + b**2
    c.backward()
    assert abs(a.grad - 6.0) < 1e-12, f"a.grad={a.grad}"
    assert abs(b.grad - 8.0) < 1e-12, f"b.grad={b.grad}"
    print(f"Q5 ✅  c=a²+b²: ∂c/∂a={a.grad:.1f}=2a=6, ∂c/∂b={b.grad:.1f}=2b=8")
except (NotImplementedError, TypeError):
    print(f"Q5 ✅  c=a²+b²: ∂c/∂a=2×3=6, ∂c/∂b=2×4=8（算子待实现）")
print("\n🎉 算子梯度白板挑战通过！")

In [ ]:
# ✏️ 本课自评
l55_review = {
    "pow_backward":            None,  # 记住 d(x^n)/dx=n·x^(n-1)，正确实现_backward？True/False
    "relu_subgradient":        None,  # 理解 relu 在 x=0 用次梯度 0 的原因？True/False
    "tanh_forward_trick":      None,  # 记住 tanh'=1-tanh²，用 out.data 直接计算？True/False
    "all_ops_implemented":     None,  # __pow__/relu/tanh 全部实现且感知机backward通过？True/False
    "whiteboard_passed":       None,  # 白板挑战 5 道手推全部完成？True/False
}

unfilled = [k for k, v in l55_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l55_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L55 全部通关！进入 L56：反向传播手推')

---

→ **下一课**　[L56 · 反向传播手推](L56_backward_pass.ipynb)

> 下节课将学习 **反向传播手推**：链式法则逐层展开，梯度 = 局部梯度 × 上游梯度。